RAG SYSTEMS USING FAISS

In [2]:
#libraries
import os
from dotenv import load_dotenv
import numpy as np

# to make output cleaner ignoring warning
import warnings
warnings.filterwarnings('ignore')

# Langchain core imports
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate,PromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from langchain_core.messages import HumanMessage,AIMessage

# Langchain specific imports
from langchain_text_splitters import RecursiveCharacterTextSplitter
# llm for chat
from langchain_groq import ChatGroq
# model for embeddings
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_community.document_loaders import TextLoader,PyPDFLoader
from langchain_classic.chains import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain

#loading dotenv
load_dotenv()

True

Data Ingestion and Processing

In [3]:
documents = [
    Document(
        page_content="""
Artificial Intelligence (AI) is a branch of computer science focused on creating systems that can perform tasks requiring human intelligence.
These tasks include reasoning, problem-solving, decision-making, and learning from data.
AI powers applications such as virtual assistants, recommendation systems, and autonomous vehicles.
Modern AI often combines algorithms, large datasets, and computational power.
It continues to transform industries including healthcare, finance, and education.
""",
        metadata={
            "source": "ai_fundamentals.pdf",
            "page": 1,
            "topic": "Artificial Intelligence"
        }
    ),

    Document(
        page_content="""
Machine Learning (ML) is a subset of AI that enables computers to learn patterns from data without explicit programming.
ML models improve their performance by analyzing examples and making predictions.
Common learning approaches include supervised, unsupervised, and reinforcement learning.
Applications range from spam detection and fraud analysis to sales forecasting.
The quality of data plays a crucial role in model accuracy and reliability.
""",
        metadata={
            "source": "ml_basics.pdf",
            "page": 3,
            "topic": "Machine Learning"
        }
    ),

    Document(
        page_content="""
Deep Learning is a specialized area of machine learning that uses multi-layered neural networks.
These networks can automatically learn complex representations from large datasets.
Deep learning excels in image recognition, speech processing, and natural language tasks.
Popular architectures include CNNs, RNNs, and Transformers.
Its success is driven by advances in computing power and data availability.
""",
        metadata={
            "source": "deep_learning_intro.pdf",
            "page": 5,
            "topic": "Deep Learning"
        }
    ),

    Document(
        page_content="""
Natural Language Processing (NLP) focuses on enabling computers to understand and generate human language.
NLP combines linguistics, machine learning, and deep learning techniques.
Applications include chatbots, language translation, sentiment analysis, and text summarization.
Modern NLP systems often rely on transformer-based architectures.
Large Language Models have significantly advanced the capabilities of NLP systems.
""",
        metadata={
            "source": "nlp_overview.pdf",
            "page": 7,
            "topic": "Natural Language Processing"
        }
    )
]
documents

[Document(metadata={'source': 'ai_fundamentals.pdf', 'page': 1, 'topic': 'Artificial Intelligence'}, page_content='\nArtificial Intelligence (AI) is a branch of computer science focused on creating systems that can perform tasks requiring human intelligence.\nThese tasks include reasoning, problem-solving, decision-making, and learning from data.\nAI powers applications such as virtual assistants, recommendation systems, and autonomous vehicles.\nModern AI often combines algorithms, large datasets, and computational power.\nIt continues to transform industries including healthcare, finance, and education.\n'),
 Document(metadata={'source': 'ml_basics.pdf', 'page': 3, 'topic': 'Machine Learning'}, page_content='\nMachine Learning (ML) is a subset of AI that enables computers to learn patterns from data without explicit programming.\nML models improve their performance by analyzing examples and making predictions.\nCommon learning approaches include supervised, unsupervised, and reinforc

text splitting

In [4]:
splitter = RecursiveCharacterTextSplitter(
    separators=[" "],
    chunk_size = 500,
    chunk_overlap = 50,
    length_function = len
)

chunks = splitter.split_documents(documents)
print(f"No of chunks created: {len(chunks)}")

No of chunks created: 4


embedding the chunks

In [5]:
# initializing the embedding model
embeddings = HuggingFaceEmbeddings(
    model_name = "sentence-transformers/all-MiniLM-L6-v2"
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [6]:
# sample embedding
test = "My name is Arya"
test2 = "My name is Pappu"
print(embeddings.embed_query(test))
print(embeddings.embed_query(test2))

[-0.027082787826657295, -0.03755572810769081, 0.012790322303771973, 0.04538574442267418, -0.02927754633128643, -0.024144886061549187, 0.12789562344551086, -0.03597508370876312, 0.09316213428974152, -0.03097255900502205, -0.07303693145513535, -0.13469457626342773, 0.07631545513868332, -0.05207642912864685, -0.02569855935871601, 0.028436286374926567, 0.0674481987953186, -0.015208491124212742, -0.08717106282711029, -0.07169880717992783, -0.12300381809473038, 0.004460286349058151, -0.0015493464889004827, -0.04674778878688812, -0.010949320159852505, -0.03618650883436203, -0.012844390235841274, 0.004971679765731096, -0.06427563726902008, -0.03122635744512081, 0.019538279622793198, -0.06264854967594147, 0.062493305653333664, 0.08168123662471771, -0.044812534004449844, 0.03514578193426132, -0.07356081902980804, -0.010530571453273296, 0.0044304258190095425, 0.05816005915403366, -0.025669049471616745, -0.0709783136844635, -0.01314039807766676, -0.04411972686648369, -0.004656913690268993, 0.01113

creating FAISS vectorstore

In [7]:
vectorstore = FAISS.from_documents(
    documents=chunks,
    embedding=embeddings
)
print(f"Vectorstore created: {vectorstore}")
print(f"No of vectors added: {vectorstore.index.ntotal}")

Vectorstore created: <langchain_community.vectorstores.faiss.FAISS object at 0x000001F192A14D70>
No of vectors added: 4


saving FAISS vectorstore locally

In [8]:
vectorstore.save_local("faiss_index")

loading FAISS vectorstore

In [9]:
loaded_vectorstore = FAISS.load_local(
    folder_path = 'faiss_index',
    embeddings=embeddings,
    allow_dangerous_deserialization=True
)
print(f"Loaded Vectorstore, total vectors: {loaded_vectorstore.index.ntotal}")

Loaded Vectorstore, total vectors: 4


similarity search in FAISS

In [10]:
test_query = "What is deep learning ?"
result = vectorstore.similarity_search(query=test_query,k=3)
print(f"Query: {test_query}")
for i,doc in enumerate(result):
    print(f"\n{i+1}. Source: {doc.metadata['source']}")
    print(f"Content: {doc.page_content[:200]}")

Query: What is deep learning ?

1. Source: deep_learning_intro.pdf
Content: Deep Learning is a specialized area of machine learning that uses multi-layered neural networks.
These networks can automatically learn complex representations from large datasets.
Deep learning excel

2. Source: ai_fundamentals.pdf
Content: Artificial Intelligence (AI) is a branch of computer science focused on creating systems that can perform tasks requiring human intelligence.
These tasks include reasoning, problem-solving, decision-m

3. Source: nlp_overview.pdf
Content: Natural Language Processing (NLP) focuses on enabling computers to understand and generate human language.
NLP combines linguistics, machine learning, and deep learning techniques.
Applications includ


similarity search with score FAISS

In [11]:
test_query = "What is NLP?"
result = vectorstore.similarity_search_with_score(query=test_query,k=3)
print(f"Query: {test_query}")
for doc,score in result:
    print(f"\nScore: {score:.3f}")
    print(f"Source: {doc.metadata['source']}")
    print(f"Content: {doc.page_content[:200]}")

Query: What is NLP?

Score: 0.709
Source: nlp_overview.pdf
Content: Natural Language Processing (NLP) focuses on enabling computers to understand and generate human language.
NLP combines linguistics, machine learning, and deep learning techniques.
Applications includ

Score: 1.614
Source: ai_fundamentals.pdf
Content: Artificial Intelligence (AI) is a branch of computer science focused on creating systems that can perform tasks requiring human intelligence.
These tasks include reasoning, problem-solving, decision-m

Score: 1.639
Source: ml_basics.pdf
Content: Machine Learning (ML) is a subset of AI that enables computers to learn patterns from data without explicit programming.
ML models improve their performance by analyzing examples and making prediction


searching with metadata filtering

In [12]:
filter_dict={"topic":"Natural Language Processing"}
filtered_result=vectorstore.similarity_search(
    query=test_query,
    k=3,
    filter=filter_dict
)
print(filtered_result)

[Document(id='98a151b0-3476-4c6c-9b6b-a19265567d17', metadata={'source': 'nlp_overview.pdf', 'page': 7, 'topic': 'Natural Language Processing'}, page_content='Natural Language Processing (NLP) focuses on enabling computers to understand and generate human language.\nNLP combines linguistics, machine learning, and deep learning techniques.\nApplications include chatbots, language translation, sentiment analysis, and text summarization.\nModern NLP systems often rely on transformer-based architectures.\nLarge Language Models have significantly advanced the capabilities of NLP systems.')]


RAG Chain with LCEL using FAISS

LLM 

In [24]:
from langchain.chat_models import init_chat_model
os.environ['GROQ_API_KEY'] = os.getenv("GROQ_API_KEY")
llm = init_chat_model(model="groq:llama-3.1-8b-instant")
llm

ChatGroq(output_version=None, profile={'max_input_tokens': 131072, 'max_output_tokens': 8192, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x000001F195774190>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000001F195774B90>, model_name='llama-3.1-8b-instant', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None)

PROMPT TEMPLATE

In [25]:
simple_prompt = ChatPromptTemplate.from_template("""Answer the question based only
on the following context:
Context: {context}
Question: {question}
Answer:""")

Converting our vectorstore to retriever

In [26]:
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k":3}
)
retriever

VectorStoreRetriever(tags=['FAISS', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x000001F192A14D70>, search_kwargs={'k': 3})

Format doc function for formatting chunks/docs for the prompt

In [27]:
from typing import List
def format_docs(docs: List[Document]) -> str:
    formatted=[]
    for i,doc in enumerate(docs):
        source = doc.metadata.get('source','unkown')
        formatted.append(f"Document {i+1} (Source: {source}):\n {doc.page_content}")
    return "\n\n".join(formatted)


Building the Simple Rag Chain

In [28]:
simple_rag_chain = (
    {
        "context": retriever | format_docs,
        "question": RunnablePassthrough()
    }
    | simple_prompt
    | llm
    | StrOutputParser()
)
simple_rag_chain

{
  context: VectorStoreRetriever(tags=['FAISS', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x000001F192A14D70>, search_kwargs={'k': 3})
           | RunnableLambda(format_docs),
  question: RunnablePassthrough()
}
| ChatPromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template='Answer the question based only\non the following context:\nContext: {context}\nQuestion: {question}\nAnswer:'), additional_kwargs={})])
| ChatGroq(output_version=None, profile={'max_input_tokens': 131072, 'max_output_tokens': 8192, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0

Conversational RAG CHAIN with FAISS

Prompt templates

In [29]:
conversational_prompt = ChatPromptTemplate.from_messages([
    ("system","You are a helpful AI assitant. Use the provided context to answer the question"),
    ("placeholder","{chat_history}"),
    ("human","Context: {context}\n\nQuestion: {input}")
])

In [31]:
def create_conversational_rag():
    return(
        RunnablePassthrough.assign(
            context = lambda x: format_docs(retriever.invoke(x["input"]))
        )
        | conversational_prompt
        | llm
        | StrOutputParser()
    )

conversational_rag = create_conversational_rag()
conversational_rag

RunnableAssign(mapper={
  context: RunnableLambda(lambda x: format_docs(retriever.invoke(x['input'])))
})
| ChatPromptTemplate(input_variables=['context', 'input'], optional_variables=['chat_history'], input_types={'chat_history': list[typing.Annotated[typing.Union[typing.Annotated[langchain_core.messages.ai.AIMessage, Tag(tag='ai')], typing.Annotated[langchain_core.messages.human.HumanMessage, Tag(tag='human')], typing.Annotated[langchain_core.messages.chat.ChatMessage, Tag(tag='chat')], typing.Annotated[langchain_core.messages.system.SystemMessage, Tag(tag='system')], typing.Annotated[langchain_core.messages.function.FunctionMessage, Tag(tag='function')], typing.Annotated[langchain_core.messages.tool.ToolMessage, Tag(tag='tool')], typing.Annotated[langchain_core.messages.ai.AIMessageChunk, Tag(tag='AIMessageChunk')], typing.Annotated[langchain_core.messages.human.HumanMessageChunk, Tag(tag='HumanMessageChunk')], typing.Annotated[langchain_core.messages.chat.ChatMessageChunk, Tag(tag=

Streaming RAG Chain with FAISS

In [32]:
streaming_rag_chain = (
    {
        "context": retriever | format_docs,
        "question":RunnablePassthrough()
    }
    | simple_prompt
    | llm
)
# in streaming rag chain StrOutputPraser() is not added to the chain

TESTING ALL THE RAG CHAINS

In [33]:
def test_rag_chains(question: str):
    print(f"Question: {question}")
    print("=" * 80)

    # Simple RAG Chain LCEL
    print("\nSimple Rag Chain:")
    answer = simple_rag_chain.invoke(question)
    print(f"Answer: {answer}")


In [34]:
test_rag_chains("What is the difference between AI and Machine Learning")

Question: What is the difference between AI and Machine Learning

Simple Rag Chain:
Answer: Based on the given context, the difference between AI and Machine Learning is as follows:

Artificial Intelligence (AI) is a broader field that focuses on creating systems capable of performing tasks that require human intelligence, including reasoning, problem-solving, decision-making, and learning from data. AI powers various applications and transforms industries.

Machine Learning (ML), on the other hand, is a subset of AI that specifically enables computers to learn patterns from data without explicit programming. ML models improve their performance by analyzing examples and making predictions, and it has various learning approaches such as supervised, unsupervised, and reinforcement learning.

In other words, AI encompasses the overall field, while ML is a key component of AI that deals with the learning aspect.
